### Data

In [1]:
import pandas as pd
import numpy as np
import kagglehub

In [4]:
manhwa_dataset = kagglehub.dataset_download("artheon/manhwa-industry-evolution-2000-2026", path="manhwa_dataset_final.csv")
print("Dataset:", manhwa_dataset)

Using Colab cache for faster access to the 'manhwa-industry-evolution-2000-2026' dataset.
Dataset: /kaggle/input/manhwa-industry-evolution-2000-2026/manhwa_dataset_final.csv


In [5]:
df = pd.read_csv("/kaggle/input/manhwa-industry-evolution-2000-2026/manhwa_dataset_final.csv")

In [ ]:
df.head(10)

,id,title,year,status,genres,chapters,popularity,score,mean_score,description
0,105398,Solo Leveling,2018,FINISHED,"Action, Adventure, Fantasy",201,265076,84.0,84,In a world where awakened beings called “Hunte...
1,119257,Omniscient Reader,2020,RELEASING,"Action, Adventure, Fantasy",0,117235,86.0,86,"Back then, Dok-Ja had no idea. He had no idea ..."
2,85143,Tower of God,2010,RELEASING,"Action, Adventure, Drama, Fantasy, Mystery",0,94166,81.0,82,What do you desire? Money and wealth? Honor an...
3,86964,Bastard,2014,FINISHED,"Drama, Horror, Mystery, Psychological, Romance...",94,72159,83.0,83,"High school life is hard enough, but it’s even..."
4,100568,The Horizon,2016,FINISHED,"Adventure, Drama, Psychological",21,59288,85.0,85,A world where everything has been lost. A boy ...
5,128067,SSS-Class Revival Hunter,2020,RELEASING,"Action, Adventure, Drama, Fantasy, Supernatural",0,58500,83.0,83,"In the mysterious, RPG dungeon-like Tower, Con..."
6,140407,The Greatest Estate Developer,2021,FINISHED,"Adventure, Comedy, Fantasy",222,52614,89.0,89,When civil engineering student Su-Ho Kim falls...
7,100954,Sweet Home,2017,FINISHED,"Drama, Horror, Psychological, Supernatural, Th...",141,51581,81.0,82,"After an unexpected family tragedy, a reclusiv..."
8,126297,Teenage Mercenary,2020,RELEASING,"Action, Drama",0,51505,79.0,80,"At the age of eight, I-Jin Yu lost his parents..."
9,85141,The God of High School,2011,FINISHED,"Action, Adventure, Fantasy",569,51233,76.0,76,Mori Jin is a high school student and Taekwond...


In [6]:
df.isna().any()

,0
id,False
title,False
year,False
status,False
genres,True
chapters,False
popularity,False
score,False
mean_score,False
description,True


In [7]:
df.dropna(subset=['title', 'description'], inplace=True)
df = df[df['genres'].notna()]
df = df[df['genres'].astype(str).str.strip() != '']
df = df[df['description'].astype(str).str.len() > 20]
df['status'] = df['status'].astype(str).str.upper()
df.reset_index(drop=True, inplace=True)
df['chroma_id'] = df.index.astype(str)
print(f"Data prepared: {len(df)} items")

Data prepared: 4988 items


### Vectorization

In [8]:
!pip install chromadb -q
import chromadb
from chromadb.config import Settings

import torch
from sentence_transformers import SentenceTransformer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

In [9]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.create_collection(
    name="manhwa_collection",
    metadata={"hnsw:space": "cosine"}) # Используем косинусное расстояние

In [10]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading embedding model: {model_name}...")
embedder = SentenceTransformer(model_name)
print("Model loaded.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.


In [11]:
ids = df['chroma_id'].tolist()
documents = df['description'].tolist()
metadatas = []
for _, row in df.iterrows():
    metadatas.append({
        "title": row['title'],
        "year": int(row['year']),
        "status": row['status'],
        "genres": row['genres'],
        "score": float(row['score']),
        "popularity": int(row['popularity'])
    })

In [12]:
print("Generating embeddings...")
embeddings = embedder.encode(documents, show_progress_bar=True).tolist()

Generating embeddings...


Batches:   0%|          | 0/156 [00:00<?, ?it/s]

In [13]:
print("Adding to ChromaDB...")
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings
)

print(f"Total items: {collection.count()}")

Adding to ChromaDB...
Total items: 4988


### RAG Retrieval

In [14]:
def search_manhwa_chroma(query, top_k=5, min_year=None, max_year=None, status_filter=None, min_score=0):

    conditions = []

    # Собираем отдельные условия
    if min_year is not None:
        conditions.append({"year": {"$gte": min_year}})

    if max_year is not None:
        conditions.append({"year": {"$lte": max_year}})

    if status_filter is not None:
        conditions.append({"status": status_filter.upper()})

    if min_score > 0:
        conditions.append({"score": {"$gte": min_score}})

    # Формируем итоговый фильтр
    where_clause = None
    if len(conditions) == 0:
        where_clause = None
    elif len(conditions) == 1:
        where_clause = conditions[0]
    else:
        # Если условий несколько, оборачиваем их в $and
        where_clause = {"$and": conditions}

    # Выполняем запрос
    # n_results увеличиваем, так как фильтрация происходит ПОСЛЕ поиска векторов
    results = collection.query(
        query_texts=[query],
        n_results=top_k * 10,
        where=where_clause
    )

    # Обработка результатов
    if not results['ids'][0]:
        return pd.DataFrame()

    data = []
    for i in range(len(results['ids'][0])):
        meta = results['metadatas'][0][i]
        data.append({
            'title': meta['title'],
            'year': meta['year'],
            'status': meta['status'],
            'genres': meta['genres'],
            'score': meta['score'],
            'description': results['documents'][0][i],
            'distance': results['distances'][0][i]
        })

    df_res = pd.DataFrame(data)

    # Сортируем по релевантности (расстоянию)
    df_res = df_res.sort_values(by='distance')

    return df_res.head(top_k)

In [15]:
print("Test 1: Simple Semantic Search")
res1 = search_manhwa_chroma("hero levels up in dungeons", top_k=3)
print(res1[['title', 'score', 'distance']].to_string())

print("\nTest 2: Filtered Search")
res2 = search_manhwa_chroma("action fantasy", top_k=3, max_year=2016, status_filter='FINISHED')
print(res2[['title', 'year', 'status', 'distance']].to_string())

print("\nTest 3: High Score Romance")
res3 = search_manhwa_chroma("romance office", top_k=3, min_score=85)
print(res3[['title', 'score', 'distance']].to_string())

Test 1: Simple Semantic Search


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 33.5MiB/s]


                             title  score  distance
0         Infinite Leveling: Murim   76.0  0.559062
1        Hardcore Leveling Warrior   74.0  0.567484
2  Sangwi 0.001% Ranker-ui Gwihwan   60.0  0.617349

Test 2: Filtered Search
                     title  year    status  distance
0                    Aharu  2015  FINISHED  0.559724
1  Jisang Choeagui Sonyeon  2011  FINISHED  0.604884
2               REDRUM 327  2003  FINISHED  0.629411

Test 3: High Score Romance
                            title  score  distance
0  My Bias Gets on the Last Train   86.0  0.726117
1              Seasons of Blossom   85.0  0.787152
2                     Jibi Eopseo   85.0  0.832418


### LLM answer generation (Qwen2.5-0.5B-Instruct)

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [17]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading LLM: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16, # Используем float16 для экономии памяти
    device_map="auto" )          # Автоматически используем GPU
print("LLM loaded successfully!")

Loading LLM: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully!


In [39]:
def generate_recommendation(query, results_df):

    if results_df is None or results_df.empty:
        return "Unfortunately, I couldn't find any manhwas that match your filters and search criteria. Please try changing your settings."

    # Формируем контекст из результатов поиска
    context_items = []
    for _, row in results_df.iterrows():
        # Берем только самое важное для контекста
        item_info = (
            f"- Title: {row['title']}\n"
            f"  Year: {row['year']}, Статус: {row['status']}\n"
            f"  Rating: {row['score']}/100\n"
            f"  Genre: {row['genres']}\n"
            f"  Description: {row['description'][:300]}..." # Ограничиваем описание, чтобы не переполнить контекст
        )
        context_items.append(item_info)

    context_text = "\n\n".join(context_items)

    system_prompt = """You are an assistant specializing in manhwa recommendation.
    ### Task
    Recommend titles based on the provided context and the user's search query.
    ### Response
    1) Language of response is English
    2) Be brief but informative, explain why these titles are suitable
    3) Don't use excessing punctuation, only commas and period
    """

    user_prompt = f"""User prompt: "{query}"

    Manhwas found:
    {context_text}

    Please recommend the best options from the list above and briefly explain your choice."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Подготовка текста для модели
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(llm_model.device)

    # Генерация ответа
    generated_ids = llm_model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.95
    )

    # Декодирование и очистка ответа
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [40]:
print("\nRAG Test")
user_query = "I love love-hate dynamics, can you recommend something new, but not ongoing?"
res = search_manhwa_chroma(user_query, top_k=3)

if not res.empty:
    print("Found candidates:")
    print(res[['title', 'year', 'score']].to_string())
    print("\nLLM Recommendation:")
    answer = generate_recommendation(user_query, res)
    print(answer)
else:
    print("Nothing found for this test.")


RAG Test
Found candidates:
                                       title  year  score
0  Seven Days of Lust - Omegaverse Anthology  2017   53.0
1                               Perfect Mine  2020   59.0
2               Samgagkimbapeul Kkaneun Beob  2014   64.0

LLM Recommendation:
Based on the given context and user's preferences, I recommend the following:

**Title:** Samgagkimbapeul Kkaneun Beob

**Reason:** This title fits well within the "Drama, Psychological, Romance" genre and has a high rating (64 out of 100). It also aligns with the theme of someone feeling their true love is being withheld by others.

**Why this title works:** The story revolves around two characters who feel their true love is being withheld by others, which aligns with the theme of love triangles described in the given context. Additionally, the psychological element adds depth to the narrative, making it more engaging for potential readers. 

Please let me know if there's anything else I can assist you with!

Gradio demo

In [ ]:
import gradio as gr


def predict_manhwa(query, min_year, max_year, status, min_score):

    status_filter = None if status == "Any" else status
    min_y = None if min_year == 0 else min_year
    max_y = None if max_year == 2026 else max_year

    # Поиск в ChromaDB
    results_df = search_manhwa_chroma(
        query=query,
        top_k=3, # Берем топ-3 для рекомендации
        min_year=min_y,
        max_year=max_y,
        status_filter=status_filter,
        min_score=min_score
    )

    # Генерация ответа через LLM
    recommendation = generate_recommendation(query, results_df)

    # Формирование таблицы результатов для красивого вывода
    if results_df is not None and not results_df.empty:
        table_data = results_df[['title', 'year', 'status', 'score', 'genres']].copy()
        table_data.columns = ['Title', 'Year', 'Status', 'Score', 'Genres']
        table_markdown = table_data.to_markdown(index=False)
    else:
        table_markdown = "*No specific titles found matching all criteria.*"

    return recommendation, table_markdown

In [38]:
# Содание интерфейса

custom_css = """
.gradio-container {
    background-color: #fff0f5; /* Lavender Blush - very light pink background */
}
.primary-button {
    background-color: #ff69b4 !important; /* Hot Pink */
    border-color: #ff69b4 !important;
    color: white !important;
}
.primary-button:hover {
    background-color: #ff1493 !important; /* Deep Pink */
    border-color: #ff1493 !important;
}
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="pink"), css=custom_css) as demo:
    gr.Markdown("# 🌸 ManhwaRec Assistant")
    gr.Markdown("Find your next favorite Korean webtoon using AI! Describe what you want, and I'll recommend the best matches 💕")

    with gr.Row():
        with gr.Column(scale=1):
            inp_query = gr.Textbox(
                label="Your Request",
                placeholder="e.g., 'Find me office romance manhwa'",
                container=True
            )
            inp_status = gr.Dropdown(choices=["Any", "FINISHED", "RELEASING"], value="Any", label="Status")
            inp_min_score = gr.Slider(0, 100, value=70, step=5, label="Min Score")

            with gr.Row():
                inp_year_range = gr.Slider(2000, 2026, value=2000, step=1, label="From Year")
                inp_year_max = gr.Slider(2000, 2026, value=2026, step=1, label="To Year")

            btn = gr.Button("Find Manhwa", variant="primary")

        with gr.Column(scale=2):
            out_ai = gr.Textbox(label="AI Recommendation", lines=8, container=True)
            out_table = gr.Markdown(label="Top Matches")

    btn.click(fn=predict_manhwa, inputs=[inp_query, inp_year_range, inp_year_max, inp_status, inp_min_score], outputs=[out_ai, out_table])

demo.launch()

/tmp/ipykernel_7774/3673743919.py:18: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="pink"), css=custom_css) as demo:
/tmp/ipykernel_7774/3673743919.py:18: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="pink"), css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://01f23405e8e2475fea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
